# Biomolecules and the Chemical Basis of Life Workflow

This notebook scaffold supports the article **Biomolecules and the Chemical Basis of Life**. It can be expanded with biomolecular composition, elemental ratios, enzyme kinetics, ligand binding, sequence features, polymerization mass balance, condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
composition = pd.read_csv(article_dir / 'data' / 'biomolecule_composition.csv')
cols = ['carbohydrate_mg','lipid_mg','protein_mg','nucleic_acid_mg','metabolite_mg']
composition['total_biomolecule_mg'] = composition[cols].sum(axis=1)
for col in cols:
    composition[col.replace('_mg','_fraction')] = composition[col] / composition['total_biomolecule_mg']
composition.round(4)

In [ ]:
elements = pd.read_csv(article_dir / 'data' / 'elemental_composition.csv')
elements['C_to_N'] = elements['carbon_mmol'] / elements['nitrogen_mmol']
elements['C_to_P'] = elements['carbon_mmol'] / elements['phosphorus_mmol']
elements['N_to_P'] = elements['nitrogen_mmol'] / elements['phosphorus_mmol']
elements.round(4)

In [ ]:
assays = pd.read_csv(article_dir / 'data' / 'enzyme_assays.csv')
assays['velocity'] = assays['Vmax'] * assays['substrate_mM'] / (assays['Km'] + assays['substrate_mM'])
assays['fraction_vmax'] = assays['velocity'] / assays['Vmax']
assays.round(5)

In [ ]:
sequences = pd.read_csv(article_dir / 'data' / 'sequences.csv')
rows = []
for _, row in sequences.iterrows():
    seq = row['sequence']
    if row['sequence_type'].lower() == 'dna':
        counts = Counter(seq)
        rows.append({'sequence_id': row['sequence_id'], 'sequence_type': 'DNA', 'length': len(seq), 'gc_content': (counts.get('G',0)+counts.get('C',0))/len(seq)})
    else:
        counts = Counter(seq)
        hydrophobic = set('AILMFWYV')
        charged = set('DEKRH')
        rows.append({'sequence_id': row['sequence_id'], 'sequence_type': 'protein', 'length': len(seq), 'hydrophobic_fraction': sum(counts.get(a,0) for a in hydrophobic)/len(seq), 'charged_fraction': sum(counts.get(a,0) for a in charged)/len(seq)})
pd.DataFrame(rows).round(4)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'biomolecular_condition_sites.csv')
condition['biomolecular_condition_score'] = (
    0.14 * condition['carbohydrate_support'] +
    0.15 * condition['lipid_boundary_function'] +
    0.18 * condition['protein_function'] +
    0.17 * condition['nucleic_acid_integrity'] +
    0.14 * condition['metabolite_balance'] +
    0.12 * condition['cofactor_availability'] +
    0.10 * (1 - condition['stress_penalty'])
)
condition.sort_values('biomolecular_condition_score', ascending=False).round(3)